In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.12159556150436401
epoch:  1, loss: 0.04625333100557327
epoch:  2, loss: 0.041870277374982834
epoch:  3, loss: 0.01795492321252823
epoch:  4, loss: 0.016216229647397995
epoch:  5, loss: 0.013866608031094074
epoch:  6, loss: 0.008951382711529732
epoch:  7, loss: 0.008146855980157852
epoch:  8, loss: 0.007414850406348705
epoch:  9, loss: 0.007188622839748859
epoch:  10, loss: 0.005872158333659172
epoch:  11, loss: 0.004993066657334566
epoch:  12, loss: 0.004403744358569384
epoch:  13, loss: 0.0038376504089683294
epoch:  14, loss: 0.0034401577431708574
epoch:  15, loss: 0.0031493743881583214
epoch:  16, loss: 0.0029166038148105145
epoch:  17, loss: 0.002738405717536807
epoch:  18, loss: 0.002592778066173196
epoch:  19, loss: 0.002472934313118458
epoch:  20, loss: 0.0023708315566182137
epoch:  21, loss: 0.002281925408169627
epoch:  22, loss: 0.002201066818088293
epoch:  23, loss: 0.0021271351724863052
epoch:  24, loss: 0.0020594371017068624
epoch:  25, loss: 0.00198543979

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9984524250030518
Test metrics:  R2 = 0.995598316192627
